# Neural Network

## Step 0: Library & Data Imports
For the neural network, only two libraries were used: pandas and numpy. They are useful for a number of things, from loading the data from the csv file to matrix multiplication.

No other libraries were used in order to showcase a "from scratch" implementation that demonstrates course material comprehension.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
print(df.columns)

Index(['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')


## Step 1: Pre-Treatment

The dataset has 14 binary features, 3 quantitative features and 4 ordinal variables. While keeping the ordinality makes sense for GenHlth, the patient's rating of their general health, and Education, age and income can quite naturally be transformed back to quantitative data. The only issue was choosing a suitable number for the last, open-ended category of each. For age, 85 was chosen as an estimation of what the average age of that group might be. For income, the number was brought up a bit from the "natural progression" in order to better reflect the open-endedness.

In [2]:
age_map = {1: 21, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57, 9: 62, 10: 67, 11: 72, 12: 77, 13: 85}
income_map = {1: 5000, 2: 12500, 3: 17500, 4: 22500, 5: 30000, 6: 42500, 7: 62500, 8: 87500}
df["IncomeMid"] = df["Income"].map(income_map)
df["AgeMid"] = df["Age"].map(age_map)

print(df.columns)

Index(['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income',
       'IncomeMid', 'AgeMid'],
      dtype='object')


We also define a function to split the dataframe into a training and validation dataset, as well as standardizing the data. It is done in a function so we can quickly call it with various random seeds in order to compare the model's performance accross multiple splits of the data.

In [3]:
def splitAndStandardize(df, seed):
    """
    Splits dataframe into training + validation datasets with an 8:2 split, and standardized the data. Also converts to numpy as it is faster for matrix multiplication.
    :param df: Dataframe. Must essentially be the one we preprocessed beforehand.
    :param seed: Random seed to use for the split.
    :return: 4 numpy arrays: X_train, Y_train, X_val, Y_val.
    """
    # Beginning by splitting the dataframe into two parts, using proportions 8:2 for training:validation

    np.random.seed(seed)
    shuffled_df = df.sample(frac=1).reset_index(drop=True)
    shuffled_df = shuffled_df.drop(columns=["Income", "Age"])

    train_df = shuffled_df.iloc[:(int(len(shuffled_df) * (8 / 10)))].copy()
    val_df = shuffled_df.iloc[int(len(shuffled_df) * (8 / 10)):].copy()

    # Standardizing
    for category in ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Education", "IncomeMid", "AgeMid"]:
        mean_tr = train_df[category].mean()
        std_tr = train_df[category].std()
        train_df[category] = (train_df[category] - mean_tr) / std_tr
        val_df[category] = (val_df[category] - mean_tr) / std_tr

    # Separating Xs and Ys
    X_train = train_df.drop(columns=["Diabetes_binary"])
    Y_train = train_df["Diabetes_binary"]
    Y_train = np.asarray(Y_train).reshape(-1, 1)
    X_val = val_df.drop(columns=["Diabetes_binary"])
    Y_val = val_df["Diabetes_binary"]
    Y_val = np.asarray(Y_val).reshape(-1, 1)

    # numpy is faster for matrix multiplication, which we'll be doing tons of, so converting df into np arrays
    X_train_np = X_train.to_numpy()
    Y_train_np = np.asarray(Y_train).reshape(-1,1)
    X_val_np = X_val.to_numpy()
    Y_val_np = np.asarray(Y_val).reshape(-1,1)
    return X_train_np, Y_train_np, X_val_np, Y_val_np

## Activation functions:

The sigmoid and reLu functions as seen in the course are implemented here, along with the derivative of the reLu function. The derivative of the sigmoid function simplifies nicely and so is implemented directly in the code.

<img src="./data/sigmoidVreLu.png">

_Description_: Sigmoid function on the left and reLu function on the right.

_Source_: Artificial Intelligence: A Modern Approach, 4th Edition, 2021. Peter Norvig and Stuart J. Russell. Pearson Education. p. 803

In [4]:
def sigmoid(x):
    x = np.clip(x,-500,500)     #prevent overflow
    return 1 / (1 + np.exp(-x))
def reLu(x):
    return np.maximum(0, x)
def reLu_deriv(x):
    return (x>0).astype(float)

## Other useful function

There are a few useful functions we will be using throughout:
1. addIntercept: adds an intercept to our data
2. makeBatches: since our dataset is quite large, processing by batches allows us to optimize processing usage
3. binaryCrossEntropy: our loss function

In [5]:
def addIntercept(X):
    """
    Adds intercept to a numpy array.
    :param X: numpy array
    :return: numpy array with intercept "column"
    """
    ones = np.ones((X.shape[0], 1))
    return np.hstack((ones, X))

def makeBatches(X, Y, batchSize):
    """
    Breaks up dataset into batches of size batchSize.
    :param X: 2d numpy array containing features
    :param Y: 2d numpy array containing target
    :param batchSize: size of batch
    :return: array of batches
    """
    m = X.shape[0]
    batches = []
    for start in range(0, m, batchSize):
        end = start + batchSize
        X_batch = X[start:end]
        Y_batch = Y[start:end]
        batches.append((X_batch, Y_batch))
    return batches

def binaryCrossEntropy(y_true, y_pred):
    """
    Loss function.
    :param y_true: True labels
    :param y_pred: Predicted labels
    :return: Loss
    """
    epsilon = 1e-12
    y_pred = np.clip(y_pred, epsilon, 1-epsilon)        # to prevent overflow
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

## Basic Steps

There are 3 basic steps of a neural network: the forward propagation, the backward propagation and the loss calculation. We also define a predict function, even though it is technically just an application of the forward propagation.

In [6]:
def forwardPropagation(X_batch, W1, W2):
    """
    Forward Propagation: matrix multiplies X and W1, applies the reLu function, adds the intercept again, matrix multiplies result and W2, and then applies the sigmoid function. Puts everything into a cache and returns it.
    :param X_batch: batch of features to work on
    :param W1: weights 1 matrix
    :param W2: weights 2 matrix
    :return: NN output (A2) + cache containing intermediates and x batch.
    """
    Z1 = X_batch @ W1           # matrix multiply X and W1
    A1 = reLu(Z1)               # apply reLu function
    A1_i = addIntercept(A1)     # add intercept
    Z2 = A1_i @ W2              # matrix multiply result and W2
    A2 = sigmoid(Z2)            # apply sigmoid function
    cache = {"X_batch": X_batch, "Z1": Z1, "A1": A1, "A1_i":A1_i, "Z2": Z2, "A2": A2}
    return A2, cache


def backwardPropagation(Y_batch, W2, cache):
    """
    Backward propagation: calculate output error, then gradient for W2, error passed to hidden layer, hidden layer's error, then gradient for W1.
    :param Y_batch: batch's label
    :param W2: weights 2 matrix
    :param cache: cache containing intermediates and x batch
    :return: gradient for W1 and W2
    """
    # unpack cache
    X_batch = cache["X_batch"]
    Z1 = cache["Z1"]
    A1 = cache["A1"]
    A1_i = cache["A1_i"]
    A2 = cache["A2"]
    m = X_batch.shape[0]

    #output error
    dZ2 = A2 - Y_batch
    # gradient for w2
    dW2 = (A1_i.T @ dZ2) / m
    # error passed back to hidden layer
    dA1 = dZ2 @ W2[1:,:].T
    # hidden layer error
    dZ1 = dA1 * reLu_deriv(Z1)
    # gradient for W1
    dW1 = (X_batch.T @ dZ1) / m

    return dW1, dW2

def predict(X, W1, W2):
    """
    Predict labels using weights.
    :param X: feature matrix on which to predict
    :param W1: weights 1 matrix
    :param W2: weights 2 matrix
    :return: the labels and the probabilities (A2)
    """
    A2, cache = forwardPropagation(X, W1, W2)
    y_class = (A2 > 0.5).astype(int)
    return y_class, A2

## Optimization

We optimize:
1. Hidden layer size
2.

In [12]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def runNN(df, seed, hiddenLayerSize=30):
    X_train_np, Y_train_np, X_val_np, Y_val_np = splitAndStandardize(df,seed=seed)

    learning_rate=0.02
    epochs = 1000

    # We implement early stop when the loss no longer goes down for 30 epochs. To do so, we keep in memory the best loss encountered and the corresponding weights, and implement a patience counter which goes down each epoch that the loss does not decrease, and resets if it does.
    best_val_loss = float("inf")
    best_W1 = None
    best_W2 = None
    patience_counter = 0
    best_epoch = 0
    patience = 30           # max patience
    min_delta = 0.0001      # minimum loss change needed

    # adding intercept to our data
    X_train_np = addIntercept(X_train_np)
    X_val_np = addIntercept(X_val_np)

    # setting up size of NN
    input_size = X_train_np.shape[1]
    output_size = 1
    hidden_layer_size = hiddenLayerSize          # although not shown, multiple hidden layer sizes were tested, and this was found to be a good size

    # generating starting weights
    np.random.seed(seed)
    # W1 is the connection between input to hidden layer. shape (22,hidden_layer_size)
    W1 = np.random.randn(input_size, hidden_layer_size) * np.sqrt(2 / input_size)
    # W2 is the connection between hidden to output. shape (hidden_layer_size,1)
    W2 = np.random.randn(hidden_layer_size+1, output_size) * np.sqrt(2 / hidden_layer_size)

    n_samples = X_train_np.shape[0]
    batchSize = 128
    for epoch in range(epochs):
        indices = np.arange(n_samples)
        np.random.shuffle(indices)
        X_shuffled = X_train_np[indices]
        Y_shuffled = Y_train_np[indices]
        for start in range(0, n_samples, batchSize):
            end = start + batchSize
            X_batch = X_shuffled[start:end]
            Y_batch = Y_shuffled[start:end]
            m = X_batch.shape[0]

            A2, cache = forwardPropagation(X_batch, W1, W2)
            dW1, dW2 = backwardPropagation(Y_batch, W2, cache)

            # update weights
            W2 = W2 - learning_rate * dW2
            W1 = W1 - learning_rate * dW1

        # check current loss & accuracy on validation set
        val_class, val_pred = predict(X_val_np, W1, W2)
        val_loss = binaryCrossEntropy(Y_val_np, val_pred)
        val_acc = accuracy(Y_val_np, val_class)

        # if
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_W1 = W1.copy()
            best_W2 = W2.copy()
            patience_counter = 0
            best_epoch = epoch

        else:
            patience_counter += 1

        # print to check progress
        """if epoch % 50 == 49:
            train_class, train_pred = predict(X_train_np, W1, W2)
            train_loss = binaryCrossEntropy(Y_train_np, train_pred)
            train_acc = accuracy(Y_train_np, train_class)
            Z1_val = X_val_np @ W1
            A1_val = reLu(Z1_val)
            A1_val_i = addIntercept(A1_val)
            Z2_val = A1_val_i @ W2
            val_pred = sigmoid(Z2_val)
            val_class = (val_pred > 0.5).astype(int)

            val_loss = binaryCrossEntropy(Y_val_np, val_pred)
            val_acc = accuracy(Y_val_np, val_class)

            print(
                "Epoch:", epoch,
                "| Train loss:", round(train_loss, 4),
                "| Train acc:", round(train_acc, 4),
                "| Val loss:", round(val_loss, 4),
                "| Val acc:", round(val_acc, 4)
            )"""
        if patience_counter >= patience:
            """print("Early stopping at epoch:", epoch)
            print("Best epoch:", best_epoch)
            print("Best validation loss:", best_val_loss)
            print("Best validation accuracy:", best_val_acc)"""
            break
    return best_W1,best_W2

In [8]:
"""for hls in [10,15,20,25,30,35,40,45,50]:
    print("hls: ", hls)
    runNN(df,42,hls)"""

'for hls in [10,15,20,25,30,35,40,45,50]:\n    print("hls: ", hls)\n    runNN(df,42,hls)'

In [15]:
import time
from sklearn.metrics import roc_curve, auc
runTimes = []
runAcc = []
runAUC = []
runSpec = []
randseed = 137
dfDeepCopy = df.copy()
for i in range(10):
    df = dfDeepCopy.copy()
    print("Running... run number: ",i)
    start_time = time.time()
    SEED = randseed+i
    W1_best, W2_best = runNN(df,seed=SEED,hiddenLayerSize=30)

    X_train_np, Y_train_np, X_val_np, Y_val_np = splitAndStandardize(df,seed=randseed)

    # adding intercept to our data
    X_train_np = addIntercept(X_train_np)
    X_val_np = addIntercept(X_val_np)

    # get predicted probs on validation set
    val_class, val_prob = predict(X_val_np, W1_best, W2_best)

    # sklearn needs 1D arrays
    y_true = Y_val_np.ravel()
    y_score = val_prob.ravel()

    # compute ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    # Compute auc
    roc_auc = auc(fpr, tpr)
    # Youden's J statistic
    j_scores = tpr - fpr
    best_index = np.argmax(j_scores)

    best_threshold = thresholds[best_index]
    best_tpr = tpr[best_index]
    best_fpr = fpr[best_index]
    acc = accuracy(Y_val_np, val_class)
    thisRunTime = time.time() - start_time
    runTimes.append(thisRunTime)
    runAcc.append(acc)
    runAUC.append(roc_auc)
    runSpec.append((1 - best_fpr))
    print("Run time: ", thisRunTime)
    print("Accuracy: ", acc)
    print("AUC: ", roc_auc)
    print("Specificity: ", (1-best_fpr))

# speed for seed 42: 6s 162ms, acc: 0.7533771836763562
# speed for seed 43: 4s 741ms, acc: 0.7558526062663554
# speed for seed 44: 5s 176ms, acc: 0.7499823184100715

Running... run number:  0
Run time:  13.215141773223877
Accuracy:  0.7467289058632153
AUC:  0.8238218857111985
Specificity:  0.679926125870152
Running... run number:  1
Run time:  6.795342683792114
Accuracy:  0.751467571964071
AUC:  0.8262958086636025
Specificity:  0.7246768006819151
Running... run number:  2
Run time:  7.137859106063843
Accuracy:  0.7507603083669283
AUC:  0.8266657195624378
Specificity:  0.6745276317658758
Running... run number:  3
Run time:  11.247308015823364
Accuracy:  0.7537308154749275
AUC:  0.8279395480712088
Specificity:  0.7246768006819151
Running... run number:  4
Run time:  7.540555953979492
Accuracy:  0.7513261192446424
AUC:  0.8268261736922458
Specificity:  0.7367523795993749
Running... run number:  5
Run time:  4.834569931030273
Accuracy:  0.7524577410000707
AUC:  0.8262643141131202
Specificity:  0.7079130558317943
Running... run number:  6
Run time:  5.886618137359619
Accuracy:  0.7506188556474999
AUC:  0.8254285679984154
Specificity:  0.6929961642278732

In [16]:
runTimes = np.array(runTimes)
runAUC = np.array(runAUC)
runSpec = np.array(runSpec)
runAcc = np.array(runAcc)

print("For run times:")
std_sample = np.std(runTimes, ddof=1)
sample_mean = np.mean(runTimes)
print("Sample std:", std_sample)
print("Sample mean:", sample_mean)

print("For AUC:")
std_sample = np.std(runAUC, ddof=1)
sample_mean = np.mean(runAUC)
print("Sample std:", std_sample)
print("Sample mean:", sample_mean)

print("For specificity:")
std_sample = np.std(runSpec, ddof=1)
sample_mean = np.mean(runSpec)
print("Sample std:", std_sample)
print("Sample mean:", sample_mean)

print("For accuracy:")
std_sample = np.std(runAcc, ddof=1)
sample_mean = np.mean(runAcc)
print("Sample std:", std_sample)
print("Sample mean:", sample_mean)

For run times:
Sample std: 2.530778581324371
Sample mean: 7.739410853385925
For AUC:
Sample std: 0.0011944482175597901
Sample mean: 0.8265278438638651
For specificity:
Sample std: 0.022224103949895876
Sample mean: 0.705995169768433
For accuracy:
Sample std: 0.0018093450010971646
Sample mean: 0.7511351580734139


In [ ]:
for randseed in [42,137,514]:
    W1_best, W2_best = runNN(df,randseed,30)

    X_train_np, Y_train_np, X_val_np, Y_val_np = splitAndStandardize(df,seed=randseed)

    # adding intercept to our data
    X_train_np = addIntercept(X_train_np)
    X_val_np = addIntercept(X_val_np)

    from sklearn.metrics import roc_curve, auc
    import matplotlib.pyplot as plt

    # get predicted probs on validation set
    val_class, val_prob = predict(X_val_np, W1_best, W2_best)

    # sklearn needs 1D arrays
    y_true = Y_val_np.ravel()
    y_score = val_prob.ravel()

    # compute ROC curve
    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    # Compute auc
    roc_auc = auc(fpr, tpr)

    print("Validation AUC:", roc_auc)

    # plot ROC curve
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"Neural Network ROC curve, AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - Validation Set")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Youden's J statistic
    j_scores = tpr - fpr
    best_index = np.argmax(j_scores)

    best_threshold = thresholds[best_index]
    best_tpr = tpr[best_index]
    best_fpr = fpr[best_index]

    print("Best threshold:", best_threshold)
    print("True positive rate:", best_tpr)
    print("False positive rate:", best_fpr)
    print("Specificity:", 1 - best_fpr)